In [ ]:
import sys
from pathlib import Path

# --- Imports
import os
from collections import Counter
import seaborn as sns

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



# notebook lives in  Iracing/Notebooks/ → go up one level to reach Iracing/
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from config import TRACK_CONFIGS, DATASETS, DRIVER_ALIAS, resolve_stint_files
from config import IRSDK_AVAILABLE, IBT_CHANNELS, build_lap_validity_table, basic_clean_and_units, assign_sectors

print(f'irsdk available  : {IRSDK_AVAILABLE}')
print(f'Tracks available : {list(TRACK_CONFIGS.keys())}')
print(f'Driver aliases   : {DRIVER_ALIAS}')


# User Selection

In [ ]:
# =========================
# USER SELECTION
# =========================

TRACK = "charlotte_roval_2025"  # options: "charlotte_roval_2025", "summit_point"
track_id = TRACK  # for display and file naming

# Use real names here — aliases are applied below for display only
DRIVER_REF  = "Rodrigo"     # real name (key in DATASETS)
STINT_REF   = "stint_1"

DRIVER_TEST = "Tomaz"       # real name (key in DATASETS)
STINT_TEST  = "stint_2"


In [ ]:
# aliases resolved from shared config
driver_a  = DRIVER_ALIAS.get(DRIVER_REF,  DRIVER_REF)
driver_b  = DRIVER_ALIAS.get(DRIVER_TEST, DRIVER_TEST)
track_id  = TRACK

# file paths via shared registry (handles single & multi-file stints)
ibt_file_A = [str(p) for p in resolve_stint_files(TRACK, DRIVER_REF,  STINT_REF)]
ibt_file_B = [str(p) for p in resolve_stint_files(TRACK, DRIVER_TEST, STINT_TEST)]

print(f'Reference : {driver_a} ({DRIVER_REF}) - {STINT_REF}   -> {len(ibt_file_A)} file(s)')
print(f'Test      : {driver_b} ({DRIVER_TEST}) - {STINT_TEST}  -> {len(ibt_file_B)} file(s)')


# Import Track Sectors Edge

In [ ]:
# load track geometry from shared TRACK_CONFIGS
cfg          = TRACK_CONFIGS[TRACK]
CUSTOM_EDGES = cfg['custom_edges']
SECTOR_NAMES = cfg['sector_names']
TRACK_NAME   = cfg['track_name']

print(f'Track  : {TRACK_NAME}')
print(f'Sectors: {len(CUSTOM_EDGES) - 1}')


# 2. Data loading

Two options:

- **From `.ibt` via `irsdk`** (preferred for raw sessions).
- **From CSV** (if you already exported or `irsdk` is not available).

The notebook will try what you configured above.


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Telemetry helpers — via config package (single source of truth)
# ═══════════════════════════════════════════════════════════════
# PROJECT_ROOT e sys.path já configurados na Cell 0.
# Tudo vem de config/__init__.py → ibt_loader.py

from config import (
    load_stint,
    basic_clean_and_units,
    build_lap_validity_table,
    IBT_CHANNELS,
)

# Alias backward-compat para chamadas load_from_ibt() no resto do notebook
load_from_ibt = load_stint

print("✅ helpers importados de config → ibt_loader")

# 4. Como ibt_file_A já é uma lista de strings, passamos diretamente!
raw_df = load_from_ibt(ibt_file_A)
display(raw_df.head())

# 3) Data Cleaning



In [ ]:
from typing import Dict

def format_laptime(seconds: float) -> str:
    """Convert seconds to MM:SS.mmm string."""
    if pd.isna(seconds):
        return "N/A"
    minutes          = int(seconds // 60)
    remaining_seconds = seconds % 60
    return f"{minutes:02d}:{remaining_seconds:06.3f}"


def align_lap_by_dist(g: pd.DataFrame, grid: np.ndarray) -> Dict[str, np.ndarray]:
    g = g.sort_values("LapDistPct").drop_duplicates(subset=["LapDistPct"], keep="first")
    if g.empty:
        return {}

    t_rel = g["SessionTime"] - g["SessionTime"].iloc[0]
    x     = g["LapDistPct"].to_numpy()

    if len(x) < 2 or np.allclose(x.max() - x.min(), 0):
        return {}

    def interp(y: np.ndarray) -> np.ndarray:
        return np.interp(grid, x, y)

    return {
        "LapDistPct":        grid,
        "t_rel":             interp(t_rel.to_numpy()),
        "speed":             interp(g["Speed_KPH"].to_numpy()),
        "throttle":          interp(g["Throttle_Pct"].to_numpy()),
        "brake":             interp(g["Brake_Pct"].to_numpy()),
        "SteeringWheelAngle":interp(g["SteeringWheelAngle"].to_numpy()),
        "YawRate":           interp(g.get("YawRate",    pd.Series(np.zeros_like(x))).to_numpy()),
        "LongAccel":         interp(g.get("LongAccel",  pd.Series(np.zeros_like(x))).to_numpy()),
    }

In [ ]:
df_ref = basic_clean_and_units(raw_df)
df_ref.head()


## 4. Lap validity & reference selection

We mark laps as valid and pick a **reference lap** (fastest among valid). We'll also pick a **target lap** to compare.


In [ ]:
INVALID_LAPS = set()
TARGET_LAP_TO_ANALYZE = 4

lap_df = build_lap_validity_table(df_ref, manual_invalid=INVALID_LAPS)

lap_df["LapTime_Formatted"] = lap_df["LapTime_s"].apply(format_laptime)

display(lap_df[["Lap", "Valid", "LapTime_s", "LapTime_Formatted"]])

valid_laps = lap_df[lap_df["Valid"]]["Lap"].tolist()

if not valid_laps:
    raise RuntimeError("Nenhuma volta válida encontrada.")

# --- Referência: volta mais rápida válida ---
ref_row = lap_df[lap_df["Valid"]].sort_values("LapTime_s").iloc[0]
ref_lap = int(ref_row["Lap"])

# --- Target: tentativa manual com fallback ---
target_lap = TARGET_LAP_TO_ANALYZE

if target_lap not in valid_laps:
    print(
        f"⚠️ Volta alvo {target_lap} não existe ou é inválida.\n"
        f"Usando a volta válida mais lenta como fallback."
    )
    target_lap = max(valid_laps)

target_row = lap_df.loc[lap_df["Lap"] == target_lap].iloc[0]

print(f"Reference lap: {ref_lap}  ({format_laptime(ref_row['LapTime_s'])})")
print(f"Target lap   : {target_lap}  ({format_laptime(target_row['LapTime_s'])})")


# Import other driver telemetry

In [ ]:
raw_df_test = load_from_ibt(ibt_file_B)
#Data Cleaning
df_test = basic_clean_and_units(raw_df_test)
df_test.head()

In [ ]:
#Cleaning lap time
INVALID_LAPS = set()  # add warm-up, pit-in/out laps if needed
target_lap = 10 # Tentativa de usar a volta desejada


#retorna um DataFrame com a coluna 'LapTime_s'
lap_df = build_lap_validity_table(df_test, manual_invalid=INVALID_LAPS)

# 1. Adicionamos uma nova coluna ao DataFrame com o tempo formatado
lap_df['LapTime_Formatted'] = lap_df['LapTime_s'].apply(format_laptime)

valid_laps = lap_df[lap_df["Valid"]]["Lap"].tolist()

# Exibimos o DataFrame com a nova coluna para melhor visualização
display(lap_df[['Lap', 'Valid', 'LapTime_s', 'LapTime_Formatted']])

if not valid_laps:
    raise RuntimeError("Nenhuma volta válida encontrada para este piloto.")

# Volta de referência (mais rápida válida)
ref_row_test = lap_df[lap_df["Valid"]].sort_values("LapTime_s").iloc[0]
ref_lap = int(ref_row_test["Lap"])


if target_lap not in valid_laps:
    print(
        f"⚠️ Volta alvo {target_lap} não disponível para este piloto.\n"
        f"Usando a volta válida mais lenta como fallback."
    )
    target_lap = max(valid_laps)
    
target_row = lap_df[lap_df["Lap"] == target_lap].iloc[0]

print(f"Reference lap: {ref_lap}  ({format_laptime(ref_row['LapTime_s'])})")
print(f"Target lap   : {target_lap}  ({format_laptime(target_row['LapTime_s'])})")


# Tratamento dos Dados

In [ ]:


# Optional irsdk (for .ibt)
try:
    import irsdk
    IRSDK_AVAILABLE = True
except Exception:
    IRSDK_AVAILABLE = False

print(f"irsdk available   : {IRSDK_AVAILABLE}")

# --- Paths & Parameters
USE_IBT = True  # set True to read a .ibt with irsdk if available

# --- Configuration ---
COL_TIME = 'SessionTime'      
COL_LAP = 'Lap'              
COL_VAR = 'Throttle_Pct'  

# Core pipeline settings
N_SECTORS = 4               # number of sectors to split the track
BASE_GRID_LEN = 1000
TOPK_PERCENT = 0.30          # robust sector reference: median of fastest top K%
PACE_THRESHOLD_MS = 3000.0   # drop laps whose sum of positive sector losses > 2s
CLIP_MS = 1500.0             # clip training target (ms)
SEQ_LEN = 128                # per-sector resampling for stability
SEED = 42



# Análise das propriedades estatísticas

nesta seção serão analisados os dados das principais variáveis disponíveis no dataset, dentre elas estão:

* Posição do pedal do acelerador (%)
* Posição do pedal do freio (%)
* Tempo de volta
* Aceleração total

In [ ]:
# =========================================================
# PROCESSAMENTO E LIMPEZA DE VOLTAS (REF vs TEST)
# =========================================================

# 1. Limpeza básica e conversão de unidades
df_ref_proc  = basic_clean_and_units(df_ref)
df_test_proc = basic_clean_and_units(df_test)

# 2. Geração das tabelas de validade nomeadas
# Aqui renomeamos para identificar quem é quem
# Aplicamos a regra de exclusão APENAS para o piloto de referência (Driver A)
laps_ref = build_lap_validity_table(df_ref_proc, manual_invalid={}, max_lap_time_s=115.0)

# O piloto de teste (Driver B) continua com a validação padrão, sem descartes manuais
laps_test = build_lap_validity_table(df_test_proc, max_lap_time_s=115.0)

# Adicionamos a coluna formatada para os relatórios da tese
laps_ref['LapTime_Formatted']      = laps_ref['LapTime_s'].apply(format_laptime)
laps_test['LapTime_Formatted'] = laps_test['LapTime_s'].apply(format_laptime)

# 3. CRIAÇÃO DOS DATAFRAMES "CLEAN" (O combustível para a análise estocástica)
# Filtramos o DataFrame original para manter APENAS as linhas de voltas válidas
df_ref_clean = df_ref_proc[
    df_ref_proc["Lap"].isin(laps_ref[laps_ref["Valid"]]["Lap"])
].copy()

df_test_clean = df_test_proc[
    df_test_proc["Lap"].isin(laps_test[laps_test["Valid"]]["Lap"])
].copy()

# --- Relatório de Conferência ---
print(f"✅ {driver_a}: {len(laps_ref[laps_ref['Valid']])} voltas válidas para análise.")
print(f"✅ {driver_b}: {len(laps_test[laps_test['Valid']])} voltas válidas para análise.")

display(laps_test[laps_test["Valid"]][['Lap', 'LapTime_Formatted']].head())

In [ ]:
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def analyze_discrete_stationarity(df, lap_col, time_col):
    """
    Export-ready version: computes WSS statistics and
    returns the figure for external saving.
    """
    # 1. Data preparation
    lap_times = df.groupby(lap_col)['LocalLapTime'].max()
    laps  = lap_times.index
    times = lap_times.values

    mu    = np.mean(times)
    sigma = np.std(times)

    def format_time(seconds, _=None):
        m  = int(seconds // 60)
        s  = int(seconds % 60)
        ms = int((seconds - int(seconds)) * 1000)
        return f"{m:02d}:{s:02d}.{ms:03d}"

    # 2. Figure creation
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # --- Subplot 1: Discrete Process X_n ---
    ax1.plot(laps, times, marker='o', linestyle='-', color='royalblue',
             label=r'Lap Time $X_n$')
    ax1.axhline(mu, color='red', linestyle='--', linewidth=2,
                label=f'Mean ({format_time(mu)})')
    ax1.fill_between(laps, mu - sigma, mu + sigma, color='red', alpha=0.1,
                     label=r'$\pm 1 \sigma$')

    ax1.yaxis.set_major_formatter(FuncFormatter(format_time))
    ax1.set_title('Discrete Process: Lap Times', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Lap Number (n)', fontsize=12)
    ax1.set_ylabel('Lap Time (mm:ss.ms)', fontsize=12)
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.6)

    # --- Subplot 2: PDF ---
    sns.histplot(times, kde=True, ax=ax2, bins=10, color='royalblue', edgecolor='black')
    ax2.set_title('Probability Density Function (PDF)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Lap Time (s)', fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()

    # Return figure and statistics for external use
    return fig, mu, sigma

## Statísticas do tempo de volta

In [ ]:
# Certifique-se de que df_ref_clean e df_test_clean já foram carregados
pilotos_analise = [
    (driver_a,  STINT_REF,  df_ref_clean), 
    (driver_b, STINT_TEST, df_test_clean)
]

# 2. Pré-processamento Robusto (Garante que todas as colunas necessárias existam)
for label, stint, df in pilotos_analise:
    if df is not None and not df.empty:
        print(f"🔄 Preparando dados: {label} ({stint})")
        
        # Ordenação cronológica fundamental para Séries Temporais
        df.sort_values(['Lap', 'SessionTime'], inplace=True)
        
        # Cálculo do Tempo Local da Volta (LocalLapTime)
        # Necessário para alinhar as voltas em 0s nos gráficos de Ciclo-Estacionariedade
        df['LocalLapTime'] = df.groupby('Lap')['SessionTime'].transform(lambda x: x - x.min())
        
        # Cálculo da Magnitude de G (TotalAccel_G)
        # Usada nos diagramas G-G e análise de Entropia
        if 'TotalAccel_G' not in df.columns:
            if 'LatAccel_G' in df.columns and 'LongAccel_G' in df.columns:
                df['TotalAccel_G'] = (df['LatAccel_G']**2 + df['LongAccel_G']**2)**0.5
            else:
                # Caso os dados estejam em m/s^2 (padrão iRacing)
                df['TotalAccel_G'] = ((df['LatAccel']/9.81)**2 + (df['LongAccel']/9.81)**2)**0.5

print("\n✅ 'pilotos_analise' definido e dados sincronizados!")

In [ ]:
def compare_cyclostationarity(df_ref, df_test, target_col, filter_zeros=False, clip_range=None):
    """
    Comparative dashboard with auto y-limits based on the 99.9th percentile
    to prevent G-force spikes from compressing the driving visualization.
    """
    # 1. SMART AXIS LIMITS
    all_values = pd.concat([df_ref[target_col], df_test[target_col]])

    # 99.9th percentile to ignore impact spikes (curbs/crashes)
    v_min  = all_values.min()
    v_max  = all_values.quantile(0.999)

    # Add a breathing margin
    margin       = (v_max - v_min) * 0.1
    unified_ylim = (max(0, v_min - (margin if v_min > 0 else 0)), v_max + margin)

    # 2. PLOT SETUP
    fig, axes = plt.subplots(2, 2, figsize=(20, 15), gridspec_kw={'height_ratios': [2, 1]})

    drivers = [
        (df_ref,  driver_a, STINT_REF,  0),
        (df_test, driver_b, STINT_TEST, 1),
    ]

    for df, name, stint, col in drivers:
        if df is None or df.empty:
            continue

        # --- TOP PLOT: Ensemble Mean ---
        ax_top = axes[0, col]
        sns.lineplot(data=df, x='LocalLapTime', y=target_col, hue='Lap',
                     palette='viridis', linewidth=0.4, alpha=0.3, legend=None, ax=ax_top)

        # Ensemble mean
        df_calc        = df.copy()
        df_calc['TimeBin'] = df_calc['LocalLapTime'].round(1)
        ensemble_mean  = df_calc.groupby('TimeBin')[target_col].mean()

        ax_top.plot(ensemble_mean.index, ensemble_mean.values, color='red',
                    linewidth=2.5, label=r'Ensemble Mean $E[X(t)]$')

        # Apply synchronized auto-zoom
        ax_top.set_ylim(unified_ylim)
        ax_top.set_title(f'{name} ({stint})\n{target_col}: Cyclo-Stationary Profile',
                         fontsize=15, fontweight='bold')
        ax_top.set_xlabel('Time in Lap (s)')
        ax_top.set_ylabel(f'{target_col}')
        ax_top.grid(True, linestyle=':', alpha=0.6)
        ax_top.legend(loc='upper right')

        # --- BOTTOM PLOT: PDF per Lap ---
        ax_bottom  = axes[1, col]
        plot_data  = df.copy()
        if filter_zeros:
            plot_data = plot_data[plot_data[target_col] > (v_max * 0.01)]  # 1% of max to filter noise

        sns.kdeplot(data=plot_data, x=target_col, hue='Lap', ax=ax_bottom,
                    palette='viridis', alpha=0.3, linewidth=1.2,
                    common_norm=False, legend=False, clip=clip_range)

        # Sync PDF x-axis with top plot y-axis
        ax_bottom.set_xlim(unified_ylim)
        ax_bottom.set_title('Strict-Sense Stationarity (Density)', fontsize=13)
        ax_bottom.set_xlabel(f'{target_col} value')
        ax_bottom.grid(True, linestyle=':', alpha=0.6)

    # Spacing adjustment to avoid clipping titles
    plt.subplots_adjust(top=0.92, bottom=0.08, hspace=0.3, wspace=0.2)

    # 3. SAVE
    file_name = (
    f"cyclostationarity_{target_col.lower()}"
    f"_{driver_a}_{STINT_REF}_vs_{driver_b}_{STINT_TEST}_{track_id}.png")
    fig.savefig(Path(SAVE_DIR) / file_name, dpi=300, bbox_inches='tight')


def analyze_cyclostationarity(df, target_col, filter_zeros=False, clip_range=None):
    """
    Visualizes Cyclo-Stationarity (Top) and Strict-Sense Stationarity (Bottom).

    Args:
        filter_zeros (bool): If True, filters out values <= 1.0 in the PDF plot
                             to analyze the 'active usage' shape (useful for Brakes).
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10),
                                   gridspec_kw={'height_ratios': [2, 1]})

    # --- PLOT 1: Time Series Ensemble (TOP) ---
    sns.lineplot(data=df, x='LocalLapTime', y=target_col,
                 hue=COL_LAP, palette='viridis',
                 linewidth=0.5, alpha=0.5, legend=None, ax=ax1)

    # Ensemble mean
    df_calc            = df.copy()
    df_calc['TimeBin'] = df_calc['LocalLapTime'].round(1)
    ensemble_mean      = df_calc.groupby('TimeBin')[target_col].mean()

    ax1.plot(ensemble_mean.index, ensemble_mean.values,
             color='red', linewidth=2.5, label=r'Ensemble Mean $E[X(t)]$')

    ax1.set_title(f'Cyclo-Stationary Process: {target_col} (Ensemble)', fontsize=14)
    ax1.set_xlabel('Time since Lap Start (s)', fontsize=12)
    ax1.set_ylabel(target_col, fontsize=12)
    ax1.legend(loc='upper right')
    ax1.grid(True, linestyle='--', alpha=0.6)

    # --- PLOT 2: Strict-Sense PDF (BOTTOM) ---
    plot_data    = df.copy()
    title_suffix = ""

    if filter_zeros:
        print(f"[FILTER] Removing values <= 1.0 to analyse active usage shape...")
        plot_data    = plot_data[plot_data[target_col] > 1.0]
        title_suffix = " (Active Usage Only > 1%)"

    sns.kdeplot(data=plot_data, x=target_col, hue=COL_LAP, ax=ax2,
                palette='viridis', alpha=0.3, linewidth=1.5,
                common_norm=False, legend=False,
                clip=clip_range)  # Physical sensor limit

    ax2.set_title(f'Strict-Sense Stationarity Check: PDF per Lap{title_suffix}', fontsize=14)
    ax2.set_xlabel(f'{target_col} Value', fontsize=12)
    ax2.set_ylabel('Density', fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Output directory — mirrors lap time notebook structure ─────────
PROJECT_ROOT = Path.home() / "OneDrive/Documents/GitHub/Doutorado/Racing4all"
BASE_IMG_DIR = Path(PROJECT_ROOT) / "Iracing" / "img"

COMPARISON_ID = f"{TRACK}_{driver_a}_vs_{driver_b}"
SAVE_DIR = BASE_IMG_DIR / TRACK / COMPARISON_ID / "entropy_analysis"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {SAVE_DIR}")


In [ ]:
# Lista de pilotos para processar (Referência e Teste)
pilotos_para_analise = [
    (driver_a,  STINT_REF,  df_ref_clean),
    (driver_b, STINT_TEST, df_test_clean)
]

for label, stint, df in pilotos_para_analise:
    if df is None or df.empty:
        continue
        
    print(f"📊 Analisando Estacionariedade: {label} ({stint})")
    
    # 1. Gera o gráfico usando a função atualizada
    fig, mu, sigma = analyze_discrete_stationarity(df, COL_LAP, COL_TIME)
    
    # 2. Adiciona um título superior para identificar o piloto
    fig.suptitle(f"Discrete Stationarity: {label} - {TRACK_NAME}\nStint: {stint}", 
                 fontsize=16, fontweight='bold', y=1.02)
    
    # 3. Define o caminho de salvamento
    file_name = f"stationarity_{label.lower()}_{stint.lower()}_{track_id}.png"
    save_path = SAVE_DIR / file_name
    
    # Garante que a pasta existe
    save_path.parent.mkdir(parents=True, exist_ok=True)
    
    # 4. Salva antes de mostrar (para não perder o objeto da figura)
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✅ Gráfico salvo em: {save_path}")
    
    # 5. Exibe no Jupyter
    plt.show()
    
    # Print dos resultados numéricos no console
    print(f"   > Média: {mu:.3f}s | Desvio Padrão: {sigma:.3f}s\n")

## Análise ciclo-estacionária das principais variáveis

### - Posição do Pedal do Acelerador (%)

In [ ]:
# --- EXECUÇÃO PARA O ACELERADOR ---
compare_cyclostationarity(
    df_ref_clean, 
    df_test_clean, 
    target_col='Throttle_Pct', 
    clip_range=(0, 100)
)

### Posição do pedal de freio (%)

In [ ]:
# --- EXECUÇÃO PARA O FREIO (BRAKE) ---
# 'filter_zeros=True' é fundamental aqui para não enviesar a PDF com o tempo em reta
compare_cyclostationarity(
    df_ref_clean, 
    df_test_clean, 
    target_col='Brake_Pct', 
    filter_zeros=True, 
    clip_range=(0, 100)
)

### Velocidade (km/h)

In [ ]:
# --- EXECUÇÃO PARA VELOCIDADE ---
# Definimos o limite superior dinamicamente com base no valor máximo do dataset
max_speed = max(df_ref_clean['Speed_KPH'].max(), df_test_clean['Speed_KPH'].max())

compare_cyclostationarity(
    df_ref_clean, 
    df_test_clean, 
    target_col='Speed_KPH', 
    filter_zeros=False, 
    clip_range=(0, max_speed + 10)
)

### Tempo de Volta

#### Autocorrelação

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
from pandas.plotting import lag_plot
import matplotlib.pyplot as plt

def analyze_lap_autocorrelation(df, lap_col, time_col):
    """
    Analisa a dependência entre voltas consecutivas (Autocorrelação).
    Versão atualizada para retornar a figura e permitir salvamento externo.
    """
    # 1. Cálculo das durações das voltas (X_n)
    # Filtramos apenas as voltas válidas se a coluna 'Valid' existir (vinda do build_lap_validity)
    # Caso contrário, calculamos de todas.
    lap_durations = df.groupby(lap_col)[time_col].apply(lambda x: x.max() - x.min())
    
    # Removemos possíveis NaNs ou zeros que quebram a ACF
    lap_durations = lap_durations[lap_durations > 10] 

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # --- CHART 1: Autocorrelation Function (ACF) ---
    # Lags limitado ao número de voltas disponíveis
    max_lags = min(len(lap_durations) - 1, 15)
    plot_acf(lap_durations, ax=ax1, lags=max_lags, alpha=0.05, title='Autocorrelation Function (ACF)')
    ax1.set_xlabel('Lag (Atraso em Voltas)', fontsize=12)
    ax1.set_ylabel('Coeficiente de Correlação', fontsize=12)
    ax1.grid(True, linestyle='--', alpha=0.6)
    
    # --- CHART 2: Lag Plot (Scatter: Lap n vs Lap n+1) ---
    lag_plot(lap_durations, lag=1, ax=ax2, c='royalblue', alpha=0.7)
    
    ax2.set_title(r'Lag Plot ($X_n$ vs $X_{n+1}$)', fontsize=14)
    ax2.set_xlabel(r'Tempo Volta $X_n$ (s)', fontsize=12)
    ax2.set_ylabel(r'Tempo Próxima Volta $X_{n+1}$ (s)', fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.6)
    
    # Linha de Identidade para referência visual
    min_val, max_val = lap_durations.min(), lap_durations.max()
    ax2.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.5, label='Consistência Ideal')
    ax2.legend()

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    return fig # Essencial para o salvamento externo

# =========================================================
# EXECUÇÃO COMPARATIVA (REF vs TEST)
# =========================================================

# Usamos os DataFrames que passaram pelo filtro de voltas válidas (_clean)
pilotos_analise_temporal = [
    (driver_a,  STINT_REF,  df_ref_clean),
    (driver_b, STINT_TEST, df_test_clean)
]

for label, stint, df_clean in pilotos_analise_temporal:
    if df_clean is None or df_clean.empty:
        print(f"⚠️ Dados insuficientes para {label}")
        continue
        
    print(f"📸 Analisando Autocorrelação: {label} ({stint})")
    
    # Chama a função com o nome original
    fig = analyze_lap_autocorrelation(df_clean, COL_LAP, COL_TIME)
    
    # Adiciona título superior customizado
    fig.suptitle(f"Autocorrelação Temporal: {label} - {TRACK_NAME} ({stint})", 
                 fontsize=16, fontweight='bold', y=1.02)
    
    # Define caminho e salva antes do show()
    file_name = f"autocorrelation_{label.lower()}_{stint.lower()}_{track_id}.png"
    save_path = SAVE_DIR / file_name
    
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✅ Gráfico salvo em: {save_path}")
    
    plt.show()

# Acelerações Logitudinais e Laterais

Estão disponíveis no dataset de corrida os valores de acelerações longitudinais (eixo x do carro) e laterais (eixo y). Os valores máximos de acelerações estão limitados dentro do chamado círculo G-G que é função da dinâmica veícular do veículos. Alguns fatores são pertinentes para suportar maiores forças G's:

* Tipo de pneu
* Arquitetura de suspensão
* peso do veículo

Para calcularmos a "aceleração geral" do veículo, será utilizado a soma vetorial das duas componentes, sendo assim:

$Acc = \sqrt{A_x^2 + A_y^2}$

In [ ]:
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

def analyze_gg_dynamics(df, label, stint, lat_col='LatAccel_G', long_col='LongAccel_G'):
    """
    Visualiza o Círculo de Fricção (G-G) e a Distribuição de Carga Total.
    Mantém o nome original, mas utiliza Scatter Plot com transparência para 
    revelar a densidade estocástica da pilotagem.
    """
    df = df.copy()
    
    # 1. Garantia de Unidades (m/s² para G)
    for col, axis in [(lat_col, 'LatAccel'), (long_col, 'LongAccel')]:
        if col not in df.columns and axis in df.columns:
            df[col] = df[axis] / 9.81
            
    # Cálculo da Aceleração Resultante (Carga Total)
    total_col = 'TotalAccel_G'
    df[total_col] = np.sqrt(df[lat_col]**2 + df[long_col]**2)

    # 2. Configuração do Layout
    fig = plt.figure(figsize=(18, 8))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.2])

    # --- PLOT 1: Diagrama G-G (Scatter Cloud) ---
    ax1 = fig.add_subplot(gs[0])
    ax1.set_facecolor("#fffafa") # Fundo escuro para destacar a nuvem
    
    # Plot de dispersão com alta transparência (alpha) para ver a densidade
    ax1.scatter(df[lat_col], df[long_col], alpha=0.08, s=3, color='red', rasterized=True)
    
    # Círculos de Referência (Envelope de Grip)
    for g_val, color, ls in [(1.0, 'white', '--'), (1.5, 'yellow', ':')]:
        circle = plt.Circle((0, 0), g_val, color=color, fill=False, 
                            linestyle=ls, alpha=0.5, label=f'{g_val} G')
        ax1.add_patch(circle)

    # Ajuste de Escala Quadrada (Crucial para G-G)
    limit = 2
    ax1.set_xlim(-limit, limit)
    ax1.set_ylim(-limit, limit)
    ax1.set_aspect('equal')
    
    ax1.axhline(0, color='gray', linestyle='-', alpha=0.3)
    ax1.axvline(0, color='gray', linestyle='-', alpha=0.3)
    
    ax1.set_title(f'Diagrama G-G: {label} ({stint})', fontsize=15, fontweight='bold')
    ax1.set_xlabel('Lateral G (Curva)', fontsize=12)
    ax1.set_ylabel('Longitudinal G (Freio/Acel)', fontsize=12)
    ax1.legend(loc='upper right', facecolor='black', labelcolor='white', framealpha=0.6)

    # --- PLOT 2: Distribuição de Carga Total ---
    ax2 = fig.add_subplot(gs[1])
    sns.histplot(df[total_col], bins=60, kde=True, ax=ax2, color='crimson', stat='density', alpha=0.6)
    
    # Pico de Grip (Percentil 99.5 para evitar ruído de impacto)
    max_g = df[total_col].quantile(0.995)
    ax2.axvline(max_g, color='black', linestyle='--', label=f'Pico de Grip (~{max_g:.2f} G)')
    
    ax2.set_title(f'Distribuição de Carga Combinada ($A = \sqrt{{A_x^2 + A_y^2}}$)', fontsize=15, fontweight='bold')
    ax2.set_xlabel('Aceleração Total (G)', fontsize=12)
    ax2.legend()
    ax2.grid(True, linestyle='--', alpha=0.3)
    ax2.set_xlim(0, 3.0)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    return fig

# =========================================================
# EXECUÇÃO COMPARATIVA (REF vs TEST)
# =========================================================

for label, stint, df_clean in [ (driver_a, STINT_REF, df_ref_clean), 
                                (driver_b, STINT_TEST, df_test_clean)]:
    
    if df_clean is None or df_clean.empty: continue
        
    print(f"📸 Processando Diagrama G-G: {label} ({stint})")
    
    # Chama a função (mantendo o nome original)
    fig = analyze_gg_dynamics(df_clean, label, stint)
    
    # Título Global
    fig.suptitle(f"Dinâmica de Pneus e Envelope de Grip - {TRACK_NAME}", 
                 fontsize=16, fontweight='bold', y=1.02)

    # Salvamento Padronizado
    file_name = f"gg_dynamics_{label.lower()}_{stint.lower()}_{track_id}.png"
    save_path = SAVE_DIR / file_name
    
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✅ Salvo em: {save_path}")
    
    plt.show()

In [ ]:
# --- 1. ENSURE COLUMN EXISTS (may not have been computed outside the GG function) ---
COL_VAR = 'TotalAccel_G'

for label, df in [("Ref", df_ref_clean), ("Test", df_test_clean)]:
    if COL_VAR not in df.columns:
        # If G columns already exist, compute the resultant directly
        if 'LatAccel_G' in df.columns and 'LongAccel_G' in df.columns:
            df[COL_VAR] = np.sqrt(df['LatAccel_G']**2 + df['LongAccel_G']**2)
        else:
            # Otherwise convert from m/s² first
            df[COL_VAR] = np.sqrt((df['LatAccel']/9.81)**2 + (df['LongAccel']/9.81)**2)

# --- 2. SYNCHRONIZED CYCLO-STATIONARITY COMPARISON ---
# clip_range=None to see the real dispersion of G-force peaks
compare_cyclostationarity(
    df_ref_clean,
    df_test_clean,
    target_col=COL_VAR,
    filter_zeros=False,
    clip_range=None
)

# VARIÂNCIA

In [ ]:
def plot_comparative_variance(df_ref, df_test, target_col, y_label, unit=""):
    """
    Analyses and compares local variance (uncertainty) between two stints.
    Fixed: uses lap_data[target_col] for correct interpolation.
    """
    print(f"📊 Computing Uncertainty Profile (σ) for: {target_col}")

    def get_sigma_profile(df):
        if df is None or df.empty: return None, None

        lap_times  = df.groupby('Lap')['SessionTime'].apply(lambda x: x.max() - x.min())
        common_time = np.linspace(0, lap_times.median(), 1000)

        matrix = []
        for lap in df['Lap'].unique():
            lap_data = df[df['Lap'] == lap].sort_values('SessionTime')
            if len(lap_data) > 10:
                t_zeroed      = lap_data['SessionTime'] - lap_data['SessionTime'].iloc[0]
                signal_values = lap_data[target_col].values
                interp_signal = np.interp(common_time, t_zeroed, signal_values, right=np.nan)
                matrix.append(interp_signal)

        if not matrix: return None, None
        matrix = np.array(matrix)
        return common_time, np.nanstd(matrix, axis=0)

    t_ref,  sigma_ref  = get_sigma_profile(df_ref)
    t_test, sigma_test = get_sigma_profile(df_test)

    fig, ax = plt.subplots(figsize=(15, 6))

    if sigma_ref is not None:
        ax.plot(t_ref, sigma_ref, color='steelblue',
                label=f'Std Dev {driver_a} ({STINT_REF})', alpha=0.8, linewidth=2)
        ax.fill_between(t_ref, 0, sigma_ref, color='steelblue', alpha=0.1)

    if sigma_test is not None:
        ax.plot(t_test, sigma_test, color='firebrick',
                label=f'Std Dev {driver_b} ({STINT_TEST})', alpha=0.8, linewidth=2)
        ax.fill_between(t_test, 0, sigma_test, color='firebrick', alpha=0.1)

    ax.set_title(
        f'Temporal Uncertainty Profile: {y_label} — {TRACK_NAME}',
        fontsize=14, fontweight='bold'
    )
    ax.set_xlabel('Time in Lap (s)')
    ax.set_ylabel(f'Standard Deviation (σ) [{unit}]')
    ax.legend()
    ax.grid(True, linestyle=':', alpha=0.5)

    file_name = (
    f"variance_profile_{target_col.lower()}"
    f"_{driver_a}_{STINT_REF}_vs_{driver_b}_{STINT_TEST}_{track_id}.png")
    plt.savefig(SAVE_DIR / file_name, dpi=300, bbox_inches='tight')


# =========================================================
# ALL DOCTORAL RESEARCH VARIABLES
# =========================================================

variables = [
    ('TotalAccel_G',       'Combined G-Force',   'G'),
    ('Throttle_Pct',       'Throttle',           '%'),
    ('Brake_Pct',          'Brake',              '%'),
    ('Speed_KPH',          'Speed',              'km/h'),
    ('SteeringWheelAngle', 'Steering Angle',     'deg'),
]

for col, label, unit in variables:
    if col in df_ref_clean.columns or col in df_test_clean.columns:
        plot_comparative_variance(df_ref_clean, df_test_clean, col, label, unit)

# ENTROPIA

In [ ]:
from scipy.stats import entropy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def plot_comparative_entropy(df_ref, df_test, target_col, y_label,
                             bins=20, smooth_window=15,
                             global_range=None):
    """
    Calcula e compara o perfil de Entropia de Shannon ao longo da volta.
    Usa um range global compartilhado para garantir comparabilidade entre drivers.
    """
    print(f"🧬 Calculando Entropia de Shannon ($H$) para: {target_col}")

    # ── RANGE GLOBAL ─────────────────────────────────────────────────────────
    if global_range is None:
        combined = pd.concat([df_ref[target_col], df_test[target_col]], ignore_index=True)
        v_min = np.percentile(combined, 1)
        v_max = np.percentile(combined, 99)
    else:
        v_min, v_max = global_range
    # ─────────────────────────────────────────────────────────────────────────

    def get_entropy_profile(df):
        if df is None or df.empty:
            return None, None, None

        lap_times = df.groupby('Lap')['SessionTime'].apply(
            lambda x: x.max() - x.min()
        )
        common_time = np.linspace(0, lap_times.median(), 1000)

        ensemble_matrix = []
        for lap in df['Lap'].unique():
            lap_data = df[df['Lap'] == lap].sort_values('SessionTime')
            if len(lap_data) > 10:
                t_zeroed = lap_data['SessionTime'] - lap_data['SessionTime'].iloc[0]
                signal_values = lap_data[target_col].values
                interp_signal = np.interp(
                    common_time, t_zeroed, signal_values, right=np.nan
                )
                ensemble_matrix.append(interp_signal)

        if not ensemble_matrix:
            return None, None, None

        ensemble_matrix = np.array(ensemble_matrix)

        h_profile = []
        for t_idx in range(ensemble_matrix.shape[1]):
            slice_data = ensemble_matrix[:, t_idx]
            slice_data = slice_data[~np.isnan(slice_data)]

            if len(slice_data) > 2:
                counts, _ = np.histogram(
                    slice_data, bins=bins,
                    range=(v_min, v_max),
                    density=True
                )
                counts += 1e-10
                h = entropy(counts, base=2)
                h_profile.append(h)
            else:
                h_profile.append(0)

        h_series = pd.Series(h_profile)
        h_smooth = h_series.rolling(window=smooth_window, center=True).mean().fillna(0)
        mean_signal = np.nanmean(ensemble_matrix, axis=0)

        return common_time, h_smooth, mean_signal

    t_ref,  h_ref,  m_ref  = get_entropy_profile(df_ref)
    t_test, h_test, m_test = get_entropy_profile(df_test)

    fig, ax = plt.subplots(1, 1, figsize=(12, 6))

    if h_ref is not None:
        ax.plot(t_ref,  h_ref,  color='steelblue', linewidth=2, label=f'Entropy {driver_a}')
        ax.fill_between(t_ref,  0, h_ref,  color='steelblue', alpha=0.15)
    if h_test is not None:
        ax.plot(t_test, h_test, color='firebrick', linewidth=2, label=f'Entropy {driver_b}')
        ax.fill_between(t_test, 0, h_test, color='firebrick', alpha=0.15)

    ax.set_ylabel('Entropy (Bits) — Higher = More Uncertainty', fontsize=11)
    ax.set_xlabel('Lap time (s)', fontsize=11)
    ax.set_title(
        f'Entropy Profile: {y_label}  [shared range: {v_min:.2f} – {v_max:.2f}]',
        fontsize=12, fontweight='bold'
    )
    ax.tick_params(axis='both', labelsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=11)

    plt.tight_layout()

    # Tenta usar as variáveis globais, se existirem (assumindo que foram definidas antes)
    try:
        file_name = (
            f"entropy_profile_{target_col.lower()}"
            f"_{driver_a}_{STINT_REF}_vs_{driver_b}_{STINT_TEST}_{TRACK}.png"
        )
        plt.savefig(SAVE_DIR / file_name, dpi=300, bbox_inches='tight')
    except NameError:
        print("Aviso: Variáveis de salvamento não definidas. O gráfico não será salvo no disco.")
        
    plt.show()


# =========================================================
# RANGE GLOBAL — calculado UMA VEZ a partir de TODOS os
# stints/drivers/pistas disponíveis no DATASETS
# =========================================================

def compute_global_ranges(datasets: dict, variables: list,
                          track_filter: str = None,
                          lower_pct: float = 1,
                          upper_pct: float = 99) -> dict:
    
    col_values = {col: [] for col, _ in variables}

    for track_key, track_info in datasets.items():
        if track_filter is not None and track_key != track_filter:
            continue
        
        base_path = Path(track_info["base_path"])
        for pilot, stints in track_info["sessions"].items():
            for stint_key, filenames in stints.items():
                
                # ─── AJUSTE PRINCIPAL AQUI ──────────────────────────────────
                # 1. Garante que filenames é sempre uma lista
                if isinstance(filenames, str):
                    filenames = [filenames]
                
                # 2. Cria uma lista de caminhos completos juntando o base_path
                filepaths = [base_path / f for f in filenames]
                # ────────────────────────────────────────────────────────────
                
                try:
                    # 3. Passa a lista de caminhos para a função
                    df_raw = load_from_ibt(filepaths)
                    
                    if df_raw.empty:
                        continue
                        
                    df_tmp = basic_clean_and_units(df_raw)
                    
                    if 'TotalAccel_G' not in df_tmp.columns:
                        df_tmp['TotalAccel_G'] = np.sqrt(
                            (df_tmp['LatAccel'] / 9.81) ** 2 +
                            (df_tmp['LongAccel'] / 9.81) ** 2
                        )
                    for col, _ in variables:
                        if col in df_tmp.columns:
                            col_values[col].append(df_tmp[col].dropna().values)
                            
                    print(f"  ✅ {track_key} | {pilot} | {stint_key}")
                except Exception as e:
                    print(f"  ⚠️  {track_key} | {pilot} | {stint_key} — {e}")

    global_ranges = {}
    for col, _ in variables:
        if col_values[col]:
            all_vals = np.concatenate(col_values[col])
            v_min = np.percentile(all_vals, lower_pct)
            v_max = np.percentile(all_vals, upper_pct)
            global_ranges[col] = (v_min, v_max)
            print(f"  Range '{col}': [{v_min:.3f} — {v_max:.3f}]")
        else:
            print(f"  ⚠️  Sem dados para '{col}'")

    return global_ranges


variaveis_entropia = [
    ('TotalAccel_G',       'G-Force Combined'),
    ('SteeringWheelAngle', 'Steering Angle'),
    ('Throttle_Pct',       'Throttle'),
    ('Brake_Pct',          'Brake'),
]

print("📐 Calculando ranges globais a partir de todos os stints...")

global_ranges = compute_global_ranges(
    DATASETS, 
    variaveis_entropia,
    track_filter=TRACK   # usa a variável já definida no notebook
)

# Loop de comparação — usa sempre o mesmo range fixo
for col, label in variaveis_entropia:
    if col in df_ref_clean.columns:
        plot_comparative_entropy(
            df_ref_clean, df_test_clean,
            col, label,
            global_range=global_ranges.get(col)
        )

## Entropia Media

In [ ]:
# =========================================================
# ENTROPIA MÉDIA GLOBAL — calcula H_mean por driver/stint
# =========================================================
from scipy.stats import entropy as shannon_entropy
import numpy as np

def compute_mean_entropy(df, target_col, global_range, bins=20):
    v_min, v_max = global_range
    lap_times = df.groupby('Lap')['SessionTime'].apply(lambda x: x.max() - x.min())
    common_time = np.linspace(0, lap_times.median(), 1000)

    matrix = []
    for lap in df['Lap'].unique():
        lap_data = df[df['Lap'] == lap].sort_values('SessionTime')
        if len(lap_data) > 10:
            t_z = lap_data['SessionTime'] - lap_data['SessionTime'].iloc[0]
            interp = np.interp(common_time, t_z, lap_data[target_col].values, right=np.nan)
            matrix.append(interp)

    if not matrix:
        return np.nan

    matrix = np.array(matrix)
    h_profile = []
    for t_idx in range(matrix.shape[1]):
        slice_d = matrix[:, t_idx][~np.isnan(matrix[:, t_idx])]
        if len(slice_d) > 1:
            counts, _ = np.histogram(slice_d, bins=bins, range=(v_min, v_max), density=True)
            h_profile.append(shannon_entropy(counts + 1e-10, base=2))

    return np.mean(h_profile) if h_profile else np.nan


col = 'TotalAccel_G'

print("=" * 65)
print(f"ENTROPIA MÉDIA GLOBAL — {col}")
print(f"Pista: {TRACK}  |  Range: {global_ranges[col]}")
print("=" * 65)
print(f"{'Driver':<12} {'Stint':<12} {'H_mean (bits)':<16} {'Laps válidas'}")
print("-" * 58)

for track_key, track_info in DATASETS.items():
    if track_key != TRACK:
        continue

    base_path = Path(track_info["base_path"])

    for pilot, stints in track_info["sessions"].items():
        alias = DRIVER_ALIAS.get(pilot, pilot)

        for stint_key, filenames in stints.items():
            if isinstance(filenames, str):
                filenames = [filenames]
            filepaths = [base_path / f for f in filenames]

            try:
                df_raw = load_from_ibt(filepaths)
                if df_raw.empty:
                    continue
                df_tmp = basic_clean_and_units(df_raw)
                df_tmp.sort_values(['Lap', 'SessionTime'], inplace=True)
                if col not in df_tmp.columns:
                    df_tmp[col] = np.sqrt(
                        (df_tmp['LatAccel'] / 9.81) ** 2 +
                        (df_tmp['LongAccel'] / 9.81) ** 2
                    )
                validity = build_lap_validity_table(df_tmp)
                valid_laps = validity[validity['Valid']]['Lap'].tolist()
                df_clean = df_tmp[df_tmp['Lap'].isin(valid_laps)].copy()

                if df_clean.empty:
                    continue

                h_mean = compute_mean_entropy(df_clean, col, global_ranges[col])
                print(f"{alias:<12} {stint_key:<12} {h_mean:<16.4f} {len(valid_laps)}")

            except Exception as e:
                print(f"{alias:<12} {stint_key:<12} ERRO: {e}")

print("=" * 65)


## ENTROPIA - HEAT MAP

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.stats import entropy
import pandas as pd
import math
vmax_global = math.log2(20)  # entropia máxima teórica: distribuição uniforme em 20 bins

def plot_comparative_track_entropy(df_ref, df_test, target_col='TotalAccel_G',
                                   global_range=None):
    """
    Gera um mapa comparativo de entropia lado a lado com sinalização de Start/Finish.
    Usa range global compartilhado para garantir comparabilidade entre drivers.
    """
    fig, axes = plt.subplots(1, 2, figsize=(22, 10))

    print("⏳ Calculando e mapeando entropia espacial...")

    # ── RANGE GLOBAL ─────────────────────────────────────────────────────────
    if global_range is None:
        combined = pd.concat([df_ref[target_col], df_test[target_col]], ignore_index=True)
        v_min = np.percentile(combined, 1)
        v_max = np.percentile(combined, 99)
    else:
        v_min, v_max = global_range
    # ─────────────────────────────────────────────────────────────────────────

    def get_data_for_map(df, driver_name=""):
        lap_times   = df.groupby('Lap')['SessionTime'].apply(lambda x: x.max() - x.min())
        common_time = np.linspace(0, lap_times.median(), 1000)

        # ── Entropy ensemble matrix ───────────────────────────────────────
        matrix = []
        for lap in df['Lap'].unique():
            lap_data = df[df['Lap'] == lap].sort_values('SessionTime')
            if len(lap_data) > 10:
                t_z      = lap_data['SessionTime'] - lap_data['SessionTime'].iloc[0]
                interp_s = np.interp(common_time, t_z, lap_data[target_col], right=np.nan)
                matrix.append(interp_s)

        matrix    = np.array(matrix)
        h_profile = []
        for t_idx in range(matrix.shape[1]):
            slice_d = matrix[:, t_idx][~np.isnan(matrix[:, t_idx])]
            if len(slice_d) > 1:
                counts, _ = np.histogram(slice_d, bins=20,
                                         range=(v_min, v_max),
                                         density=True)
                h_profile.append(entropy(counts + 1e-10, base=2))
            else:
                h_profile.append(0)

        h_smooth = (pd.Series(h_profile)
                      .rolling(window=15, center=True)
                      .mean()
                      .fillna(0)
                      .values)

        # ── Lap selection via ibt_loader helper ──────────────────────────
        # fastest_lap → used for entropy label / stats
        
        validity    = build_lap_validity_table(df, verbose=False)
        valid_rows  = validity[validity["Valid"]]
        if valid_rows.empty:
            raise RuntimeError(f"[{driver_name}] No valid laps found.")
        fastest_lap = int(valid_rows.sort_values("LapTime_s").iloc[0]["Lap"])
        fastest_t   = valid_rows.sort_values("LapTime_s").iloc[0]["LapTime_s"]
        mins, secs  = int(fastest_t // 60), fastest_t % 60
        print(f"  [{driver_name}]  🏁 Geometry source: Lap {fastest_lap}  ({mins:02d}:{secs:06.3f})  ← fastest valid lap")

        best_lap_data = df[df['Lap'] == fastest_lap].sort_values('SessionTime').copy()

        # Interpolate residual NaN GPS samples and drop unfixable rows
        best_lap_data['Lat'] = best_lap_data['Lat'].interpolate(
            method='linear', limit_direction='both')
        best_lap_data['Lon'] = best_lap_data['Lon'].interpolate(
            method='linear', limit_direction='both')
        best_lap_data = best_lap_data.dropna(subset=['Lat', 'Lon'])

        t_lap       = best_lap_data['SessionTime'] - best_lap_data['SessionTime'].iloc[0]
        lat_interp  = np.interp(common_time, t_lap, best_lap_data['Lat'])
        lon_interp  = np.interp(common_time, t_lap, best_lap_data['Lon'])

        return lon_interp, lat_interp, h_smooth, best_lap_data

    lon_r, lat_r, h_r, lap_r = get_data_for_map(df_ref,  driver_name=driver_a)
    lon_t, lat_t, h_t, lap_t = get_data_for_map(df_test, driver_name=driver_b)

    # Colorbar unificada pelo máximo global entre os dois drivers
    #vmax_global = max(h_r.max(), h_t.max()) deixei fora do loop para usar o range global já calculado

    pilotos_data = [
        (lon_r, lat_r, h_r, lap_r, driver_a),
        (lon_t, lat_t, h_t, lap_t, driver_b),
    ]

    for i, (lon, lat, h_vals, lap_data, name) in enumerate(pilotos_data):
        ax = axes[i]

        ax.plot(lap_data['Lon'], lap_data['Lat'], color='silver',
                alpha=0.3, linewidth=8, zorder=1)

        sc = ax.scatter(lon, lat, c=h_vals, cmap='inferno', s=45,
                        vmin=0, vmax=vmax_global, zorder=2)

        p0     = lap_data.iloc[0]
        p_next = lap_data.iloc[8]
        lon_0, lat_0 = p0['Lon'], p0['Lat']
        dx, dy = p_next['Lon'] - lon_0, p_next['Lat'] - lat_0
        mag    = np.sqrt(dx**2 + dy**2)
        dx, dy = dx / mag, dy / mag

        track_span_x = lap_data['Lon'].max() - lap_data['Lon'].min()
        scale = track_span_x * 0.02

        perp_dx, perp_dy = -dy * (scale * 0.7), dx * (scale * 0.7)
        ax.plot([lon_0 - perp_dx, lon_0 + perp_dx],
                [lat_0 - perp_dy, lat_0 + perp_dy],
                color='white', linewidth=4, zorder=20, solid_capstyle='round')

        arrow_end_lon = lon_0 + dx * (scale * 2.5)
        arrow_end_lat = lat_0 + dy * (scale * 2.5)
        ax.annotate('', xy=(arrow_end_lon, arrow_end_lat), xytext=(lon_0, lat_0),
                    arrowprops=dict(facecolor='white', edgecolor='black',
                                    width=3, headwidth=10),
                    zorder=20)

        ax.text(lon_0 - (track_span_x * 0.05), lat_0, 'S/F',
                fontsize=18, fontweight='bold', ha='right', va='center',
                bbox=dict(boxstyle="round", fc="white", alpha=0.9))

        ax.set_title(f"Entropy Map: {name}", fontsize=18, fontweight='bold')
        ax.axis('equal')
        ax.axis('off')

    cbar_ax = fig.add_axes([0.93, 0.2, 0.02, 0.6])
    cbar = fig.colorbar(sc, cax=cbar_ax)
    cbar.set_label('Entropy (Bits)', fontsize=18)
    cbar.ax.tick_params(labelsize=14)

    plt.suptitle(
        f"{TRACK_NAME}\n"
        f"{driver_a} ({STINT_REF}) vs {driver_b} ({STINT_TEST})",
        fontsize=22, y=0.98
    )

    file_name = (
        f"comparative_entropy_map"
        f"_{driver_a}_{STINT_REF}_vs_{driver_b}_{STINT_TEST}_{track_id}.png"
    )
    fig.savefig(SAVE_DIR / file_name, dpi=300, bbox_inches='tight')
    plt.show()


# Executa usando o range global já calculado na célula anterior
plot_comparative_track_entropy(
    df_ref_clean, df_test_clean,
    global_range=global_ranges.get('TotalAccel_G')
)


# Correlação Entropia vs Lap Time

In [ ]:

from scipy.stats import entropy, pearsonr

def plot_comparative_entropy_correlation(pilotos_lista, bins=15):
    """
    Analisa a correlação entre Entropia Global e Performance para múltiplos pilotos/stints.
    """
    all_data = []

    for label, stint, df in pilotos_lista:
        if df is None or df.empty: continue
        
        print(f"📉 Calculando métricas para: {label} ({stint})")
        
        # 1. Cálculo por Volta
        for lap in df['Lap'].unique():
            subset = df[df['Lap'] == lap]
            
            # Duração da Volta
            lap_time = subset['SessionTime'].max() - subset['SessionTime'].min()
            
            # Entropia Global (Distribuição de G Total na volta)
            v_min, v_max = 0, 3.0 # Range fixo para comparação justa
            counts, _ = np.histogram(subset['TotalAccel_G'], bins=bins, range=(v_min, v_max), density=True)
            h = entropy(counts + 1e-10, base=2)
            
            all_data.append({'Piloto': f"{label} ({stint})", 'Tempo': lap_time, 'Entropia': h})

    df_corr = pd.DataFrame(all_data)

    # 2. Limpeza de Outliers (Filtramos voltas 15% acima da mediana)
    median_t = df_corr['Tempo'].median()
    df_corr = df_corr[df_corr['Tempo'] < median_t * 1.15]

    # 3. Plotagem Comparativa
    plt.figure(figsize=(12, 8))
    sns.set_style("whitegrid")

    # Gráfico de Dispersão com Regressão Linear
    scatter = sns.lmplot(data=df_corr, x='Tempo', y='Entropia', hue='Piloto', 
                         height=7, aspect=1.3, scatter_kws={'alpha':0.5, 's':80},
                         palette=['steelblue', 'firebrick'])

    # 4. Cálculo Estatístico Individual para a Legenda
    for name in df_corr['Piloto'].unique():
        subset = df_corr[df_corr['Piloto'] == name]
        r_coef, p_val = pearsonr(subset['Tempo'], subset['Entropia'])
        print(f"📊 {name} -> r: {r_coef:.3f}, p: {p_val:.4e}")

    plt.title('Entropy vs. Performance\n(Uncertainty vs. Efficiency)', fontsize=15, fontweight='bold')
    plt.xlabel('lap Time (s) - [Effiency]', fontsize=12)
    plt.ylabel('Global Entropy (bits) - Complexity', fontsize=12)
    
    # Salvamento
    plt.savefig(SAVE_DIR / f"correlation_entropy_performance_{track_id}.png", dpi=300, bbox_inches='tight')
    plt.show()

# --- EXECUÇÃO ---
# Reutiliza a lista 'pilotos_analise' que definimos anteriormente
plot_comparative_entropy_correlation(pilotos_analise)

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import entropy, pearsonr

def generate_final_summary_table(pilotos_lista, bins=15):
    summary_rows = []

    for label, stint, df in pilotos_lista:
        if df is None or df.empty: continue
        
        # 1. Métricas de Performance
        lap_durations = df.groupby('Lap')['SessionTime'].agg(lambda x: x.max() - x.min())
        # Filtro de outliers para não sujar a média
        clean_durations = lap_durations[lap_durations < lap_durations.median() * 1.1]
        
        avg_time = clean_durations.mean()
        std_time = clean_durations.std()
        best_time = clean_durations.min()

        # 2. Métricas de Entropia e Correlação
        lap_metrics = []
        for lap in clean_durations.index:
            subset = df[df['Lap'] == lap]
            
            # Entropia Global da Volta
            counts, _ = np.histogram(subset['TotalAccel_G'], bins=bins, range=(0, 3.0), density=True)
            h = entropy(counts + 1e-10, base=2)
            lap_metrics.append({'Time': clean_durations[lap], 'Entropy': h})
        
        df_metrics = pd.DataFrame(lap_metrics)
        avg_h = df_metrics['Entropy'].mean()
        
        # Coeficiente de Correlação (r) e p-value
        r_coef, p_val = pearsonr(df_metrics['Time'], df_metrics['Entropy'])

        # 3. Organização dos dados
        summary_rows.append({
            'Piloto': label,
            'Stint': stint,
            'Best Lap (s)': f"{best_time:.3f}",
            'Avg Lap (s)': f"{avg_time:.3f}",
            'Std Dev Time (s)': f"{std_time:.3f}",
            'Avg Entropy (bits)': f"{avg_h:.4f}",
            'Corr (r)': f"{r_coef:.3f}",
            'p-value': f"{p_val:.4e}"
        })

    # Criar DataFrame final
    df_final = pd.DataFrame(summary_rows)
    return df_final


# --- 2. PRÉ-PROCESSAMENTO (Sincronização Estocástica) ---
def preprocess_ibt_dataframe(df):
    """Calcula LocalLapTime e TotalAccel_G para alinhar as voltas."""
    df = df.copy()
    df.sort_values(['Lap', 'SessionTime'], inplace=True)
    
    # Cria o cronômetro local para cada volta
    df['LocalLapTime'] = df.groupby('Lap')['SessionTime'].transform(lambda x: x - x.min())
    
    # Magnitude de G (G-G Diagram e Entropia)
    # iRacing costuma entregar em m/s^2, convertemos para G (/ 9.81)
    lat_g = df['LatAccel'] / 9.81
    lon_g = df['LongAccel'] / 9.81
    df['TotalAccel_G'] = np.sqrt(lat_g**2 + lon_g**2)


In [ ]:
# --- 1. DEFINIÇÃO DAS VOLTAS INVÁLIDAS (Exemplo) ---
# Você pode preencher este dicionário com as voltas que quer remover manualmente
# por terem sido "outlayers" ou erros de pilotagem.
INVALID_LAPS = set()

# --- 2. PIPELINE DE PROCESSAMENTO ATUALIZADO ---
all_summaries = []

print("🚀 Iniciando Pipeline com Filtro de Validez...")

for track_id, config in DATASETS.items():
    base_path = Path(config['base_path'])
    track_pilotos_analise = []
    
    print(f"\n🏁 Pista: {track_id.upper()}")
    
    for driver, stints in config['sessions'].items():
        for stint_name, file_names in stints.items():
            
            # ─── AJUSTE AQUI ────────────────────────────────────────────
            # 1. Uniformiza: se for string, transforma em lista
            if isinstance(file_names, str):
                file_names = [file_names]
            
            # 2. Cria os caminhos completos
            full_paths = [base_path / f for f in file_names]
            
            # 3. Verifica se os arquivos existem (ignora os que faltam)
            valid_paths = [p for p in full_paths if p.exists()]
            if not valid_paths:
                print(f"⚠️ Nenhum arquivo encontrado para: {driver} ({stint_name})")
                continue
            # ────────────────────────────────────────────────────────────
            
            print(f"📥 Processando: {driver} ({stint_name})...")
            
            # A. Carga e Limpeza de Unidades
            # Enviamos a lista de caminhos válidos para a função
            df_raw = load_from_ibt(valid_paths)
            if df_raw.empty: continue
            
            # Garante Speed_KPH e outros nomes de colunas necessários para a validez
            df_cleaned = basic_clean_and_units(df_raw)
            
            # B. FILTRO DE VALIDEZ (Conforme sua regra)
            # 1. Gera a tabela de validez baseada em telemetria + manual_invalid
            validity_table = build_lap_validity_table(df_cleaned, manual_invalid=INVALID_LAPS)
            
            # 2. Identifica apenas as voltas marcadas como válidas
            valid_laps_list = validity_table.loc[validity_table['Valid'], 'Lap'].tolist()
            
            # 3. Filtra o DataFrame principal
            df_final = df_cleaned[df_cleaned['Lap'].isin(valid_laps_list)].copy()
            
            if df_final.empty:
                print(f"⚠️ Nenhuma volta válida encontrada para {driver} ({stint_name})")
                continue

            # C. Pré-processamento Estocástico (LocalTime e G-Force)
            df_final = preprocess_ibt_dataframe(df_final)
            
            track_pilotos_analise.append((driver, stint_name, df_final))
            print(f"✅ {len(valid_laps_list)} voltas válidas integradas.")

    # Gera a tabela de métricas para esta pista
    if track_pilotos_analise:
        track_summary = generate_final_summary_table(track_pilotos_analise)
        track_summary['Track'] = track_id
        track_summary['Car'] = config['car']
        all_summaries.append(track_summary)

# --- 3. EXIBIÇÃO ---
if all_summaries:
    final_phd_table = pd.concat(all_summaries, ignore_index=True)
    display(final_phd_table)